# 5장 1강: A/B 테스트 설계 원리와 랜덤화 — 실습문제

## 실습 목표

- A/B 테스트의 대조군, 실험군, 랜덤화 단위를 데이터에서 식별합니다.
- 핵심 지표, 보조 지표, 가드레일 지표를 실험 목적에 맞게 사전에 정의합니다.
- 그룹 배정 수와 비율을 확인하고 계획한 50:50 배정과 일치하는지 점검합니다.
- 가설 → 설계 → 실행 → 분석의 순서로 온라인 실험계획을 작성합니다.

## 실습 환경 / 데이터

- Python, NumPy, pandas, SciPy
- `cookie_cats.csv`
- `userid`: 사용자 식별자
- `version`: 게임 게이트 위치(`gate_30`, `gate_40`)
- `sum_gamerounds`: 실험 기간 동안 플레이한 게임 라운드 수
- `retention_1`, `retention_7`: 설치 후 1일·7일 재방문 여부

> 이번 강의는 **실험 설계와 랜덤화 점검**이 중심입니다. 그룹 간 효과의 통계적 검정과 최종 배포 결정은 이후 강의에서 다룹니다.

## 실습 준비

아래 셀을 실행하여 데이터를 불러오고 크기, 결측치, 컬럼을 확인하세요.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

data_candidates = [
    Path("cookie_cats.csv"),
    Path("upload/cookie_cats.csv")]

data_path = next((path for path in data_candidates if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("cookie_cats.csv 파일을 노트북과 같은 폴더에 넣어주세요.")

df = pd.read_csv(data_path)
alpha = 0.05

print(f"데이터 크기: {df.shape[0]}행, {df.shape[1]}열")
print("전체 결측치 수:", int(df.isna().sum().sum()))
print("컬럼:", df.columns.tolist())
df.head()


데이터 크기: 90189행, 5열
전체 결측치 수: 0
컬럼: ['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7']


,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


---

## 필수 1. Cookie Cats 실험 구조와 지표 정의

게임의 첫 번째 강제 대기 게이트를 30단계에서 40단계로 옮기면 사용자 유지율이 달라지는지 확인하려고 합니다.

### 수행 요구사항

1. `userid`의 중복 여부를 확인하여 사용자 한 명이 한 행으로 기록되었는지 점검하세요.
2. `version`의 고유값과 그룹별 사용자 수를 확인하세요.
3. 버전별 사용자 수, 평균 게임 라운드 수, 1일 유지율, 7일 유지율을 하나의 요약표로 만드세요.
4. 아래 질문에 문장으로 답하세요.

### 질문

- 이 실험의 랜덤화 단위는 무엇인가요?
-> 사용자(userid) 단위입니다. 설치 시점에 각 사용자가 gate_30 또는 gate_40 중 하나에 배정되며, 한 사용자가 두 버전을 동시에 경험하지 않습니다
- `gate_30`과 `gate_40` 중 대조군과 실험군은 각각 무엇으로 설정할 수 있나요?
-> gate_30이 대조군, gate_40이 실험군입니다. gate_30이 기존에 운영되던 조건이고, 게이트를 40단계로 옮기는 것이 이번에 검증하려는 변경이기 때문입니다.
- 핵심 지표, 보조 지표, 가드레일 지표를 각각 하나씩 정하고 이유를 설명하세요.
-> 핵심 지표: 7일 유지율(retention_7). 게이트 위치 변경의 목적이 장기적인 사용자 잔존이고, 1일 유지율보다 실제 게임 지속 의향을 잘 반영합니다.

보조 지표: 1일 유지율(retention_1). 변화가 초기 이탈에서 발생하는지 아니면 며칠 후에 나타나는지 시점을 구분해 줍니다.

가드레일 지표: 평균 게임 라운드 수(sum_gamerounds). 유지율이 올라가더라도 플레이량이 크게 줄어들면 수익이나 참여도에 악영향이 있다는 신호이므로 함께 감시해야 합니다.

- 요약표에서 차이가 보인다는 사실만으로 `gate_40`의 효과라고 결론 내릴 수 있나요?
-> 없습니다. 두 가지 이유가 있습니다.

첫째, 관측된 차이(7일 유지율 0.8%p, 1일 유지율 0.6%p)가 표본 추출에 따른 우연한 변동일 가능성을 배제하지 못합니다. 같은 실험을 다시 해도 비슷한 방향의 차이가 나타날지 판단하려면 비율 검정이나 카이제곱 검정으로 유의성을 확인해야 합니다.

둘째, 차이가 통계적으로 유의하더라도 효과 크기와 실무적 의미를 따로 봐야 합니다. 표본이 9만 명 규모라 작은 차이도 유의하게 나올 수 있으므로, 신뢰구간을 구해 차이의 범위가 의사결정을 바꿀 만한 크기인지 확인하는 과정이 필요합니다.


In [3]:
# 여기에 코드를 작성하세요.
import pandas as pd

# 데이터 로드 (BOM 대비 utf-8-sig)
df = pd.read_csv("cookie_cats.csv", encoding="utf-8-sig")
df.columns = df.columns.str.strip()

# 1. userid 중복 확인
print("전체 행 수:", len(df))
print("고유 userid 수:", df["userid"].nunique())
print("중복 userid 수:", df["userid"].duplicated().sum())
print("사용자 1명 = 1행 여부:", len(df) == df["userid"].nunique())

# 2. version 고유값 및 그룹별 사용자 수
print("\nversion 고유값:", df["version"].unique().tolist())
print(df["version"].value_counts())

# 3. 버전별 요약표
summary = df.groupby("version").agg(
    users=("userid", "count"),              # 그룹별 사용자 수
    avg_rounds=("sum_gamerounds", "mean"),  # 평균 게임 라운드 수
    retention_1=("retention_1", "mean"),    # 1일 유지율
    retention_7=("retention_7", "mean"),    # 7일 유지율
).round(4)
print("\n[버전별 요약표]")
print(summary)

전체 행 수: 90189
고유 userid 수: 90189
중복 userid 수: 0
사용자 1명 = 1행 여부: True

version 고유값: ['gate_30', 'gate_40']
version
gate_40    45489
gate_30    44700
Name: count, dtype: int64

[버전별 요약표]
         users  avg_rounds  retention_1  retention_7
version                                             
gate_30  44700     52.4563       0.4482       0.1902
gate_40  45489     51.2988       0.4423       0.1820


---

## 필수 2. 무작위 배정 실습과 그룹 비율 점검

실제 `version` 값은 변경하지 않고, 동일한 사용자 목록에 연습용 A/B 그룹을 새로 무작위 배정한 뒤 실제 배정 비율과 비교하세요.

### 수행 요구사항

1. `np.random.default_rng(42)`를 사용하세요.
2. 각 `userid`에 `A` 또는 `B`를 50:50 확률로 배정한 `randomized_users`를 만드세요.
3. 연습용 그룹별 인원수와 비율을 출력하세요.
4. 실제 `version`별 인원수와 비율도 출력하세요.
5. 실제 실험이 50:50 배정을 계획했다고 가정하고, 기대빈도를 전체 인원의 절반으로 설정하여 카이제곱 적합도 검정을 수행하세요.
6. 아래 질문에 답하세요.

### 질문

- 시드를 고정하는 이유는 무엇인가요?
- 연습용 배정에서 한 사용자가 두 그룹에 동시에 포함되지 않았는지 어떻게 확인할 수 있나요?
- 실제 배정의 카이제곱 검정 결과는 50:50 계획과 일치한다고 볼 수 있나요?
- `retention_1`, `retention_7`, `sum_gamerounds`를 랜덤화 이전 공변량의 균형 점검에 사용하면 안 되는 이유는 무엇인가요?


In [5]:
# 여기에 코드를 작성하세요.

import numpy as np
import pandas as pd
from scipy import stats

df = pd.read_csv("cookie_cats.csv", encoding="utf-8-sig")
df.columns = df.columns.str.strip()

# 1~2. 시드 고정 후 연습용 A/B 무작위 배정
rng = np.random.default_rng(42)
randomized_users = df[["userid"]].copy()
randomized_users["group"] = rng.choice(["A", "B"], size=len(df), p=[0.5, 0.5])

# 3. 연습용 그룹별 인원수와 비율
prac_cnt = randomized_users["group"].value_counts().sort_index()
prac_tbl = pd.DataFrame({"n": prac_cnt, "ratio": (prac_cnt / len(randomized_users)).round(4)})
print("[연습용 배정]")
print(prac_tbl)
print("중복 userid 수:", randomized_users["userid"].duplicated().sum())
print("사용자별 그룹 개수 최대값:", randomized_users.groupby("userid")["group"].nunique().max())

# 4. 실제 version별 인원수와 비율
real_cnt = df["version"].value_counts().sort_index()
real_tbl = pd.DataFrame({"n": real_cnt, "ratio": (real_cnt / len(df)).round(4)})
print("\n[실제 배정]")
print(real_tbl)

# 5. 카이제곱 적합도 검정 (기대빈도 = 전체의 절반)
observed = real_cnt.values
expected = np.array([len(df) / 2, len(df) / 2])
chi2, p = stats.chisquare(f_obs=observed, f_exp=expected)
print("\n[카이제곱 적합도 검정: 실제 배정 vs 50:50]")
print("관측빈도:", observed, "/ 기대빈도:", expected)
print(f"chi2 = {chi2:.4f}, df = 1, p-value = {p:.4f}")
print("결론:", "50:50 계획과 다르다고 보기 어려움" if p >= 0.05 else "50:50 계획에서 벗어남")

# 참고: 연습용 배정도 동일 검정
chi2_p, p_p = stats.chisquare(f_obs=prac_cnt.values, f_exp=expected)
print(f"\n(참고) 연습용 배정: chi2 = {chi2_p:.4f}, p-value = {p_p:.4f}")


[연습용 배정]
           n   ratio
group               
A      44852  0.4973
B      45337  0.5027
중복 userid 수: 0
사용자별 그룹 개수 최대값: 1

[실제 배정]
             n   ratio
version               
gate_30  44700  0.4956
gate_40  45489  0.5044

[카이제곱 적합도 검정: 실제 배정 vs 50:50]
관측빈도: [44700 45489] / 기대빈도: [45094.5 45094.5]
chi2 = 6.9024, df = 1, p-value = 0.0086
결론: 50:50 계획에서 벗어남

(참고) 연습용 배정: chi2 = 2.6081, p-value = 0.1063


---

## 과제. Cookie Cats A/B 테스트 전체 계획서 작성

`gate_30`을 기존 버전, `gate_40`을 새 버전으로 설정한 A/B 테스트 계획을 작성하세요.

### 수행 요구사항

1. 아래 네 단계를 모두 포함한 계획서를 작성하세요.
   - 가설 설정
   - 실험 설계
   - 실험 실행
   - 결과 분석
2. 실험 단위, 대조군, 실험군, 핵심·보조·가드레일 지표를 명시하세요.
3. 실행 전에 확인할 데이터 품질 항목을 두 가지 이상 작성하세요.
4. SUTVA 위반 또는 실험 간 간섭 가능성을 검토하세요.
5. 버전별 관측 지표를 다시 계산하되, 아직 통계적 검정을 하지 않았다는 점을 반영해 최종 의사결정을 보류하는 6~8문장의 결론을 작성하세요.

> 과제는 필수 문제와 동일한 수준입니다. 표본 크기나 MDE를 계산할 필요는 없습니다.


In [7]:
# 여기에 코드를 작성하세요.

import pandas as pd

df = pd.read_csv("cookie_cats.csv", encoding="utf-8-sig")
df.columns = df.columns.str.strip()

# --- 실행 전 데이터 품질 점검 ---
print("[데이터 품질 점검]")
print("1) 중복 userid 수:", df["userid"].duplicated().sum())
print("2) 결측치 수:\n", df.isna().sum().to_string())
print("3) version 고유값:", df["version"].unique().tolist())
print("4) sum_gamerounds 범위:", df["sum_gamerounds"].min(), "~", df["sum_gamerounds"].max())
print("   상위 3개 값:", sorted(df["sum_gamerounds"], reverse=True)[:3])  # 이상치 확인
print("5) 1일 미복귀 & 7일 복귀 건수(정상 가능, 정의 확인용):",
      ((df["retention_7"]) & (~df["retention_1"])).sum())
print("6) 배정 비율:", (df["version"].value_counts(normalize=True).round(4)).to_dict())

# --- 버전별 관측 지표 재계산 ---
summary = df.groupby("version").agg(
    users=("userid", "count"),                       # 실험 단위 수
    avg_rounds=("sum_gamerounds", "mean"),           # 가드레일 지표
    median_rounds=("sum_gamerounds", "median"),      # 평균 왜곡 확인용
    retention_1=("retention_1", "mean"),             # 보조 지표
    retention_7=("retention_7", "mean"),             # 핵심 지표
).round(4)

# 절대 차이 / 상대 차이 (gate_40 - gate_30)
diff = summary.loc["gate_40"] - summary.loc["gate_30"]
rel = (diff / summary.loc["gate_30"] * 100).round(2)

print("\n[버전별 관측 지표]")
print(summary)
print("\n[차이: gate_40 - gate_30]")
print(pd.DataFrame({"절대차이": diff.round(4), "상대차이(%)": rel}).drop("users"))


[데이터 품질 점검]
1) 중복 userid 수: 0
2) 결측치 수:
 userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
3) version 고유값: ['gate_30', 'gate_40']
4) sum_gamerounds 범위: 0 ~ 49854
   상위 3개 값: [49854, 2961, 2640]
5) 1일 미복귀 & 7일 복귀 건수(정상 가능, 정의 확인용): 3599
6) 배정 비율: {'gate_40': 0.5044, 'gate_30': 0.4956}

[버전별 관측 지표]
         users  avg_rounds  median_rounds  retention_1  retention_7
version                                                            
gate_30  44700     52.4563           17.0       0.4482       0.1902
gate_40  45489     51.2988           16.0       0.4423       0.1820

[차이: gate_40 - gate_30]
                 절대차이  상대차이(%)
avg_rounds    -1.1575    -2.21
median_rounds -1.0000    -5.88
retention_1   -0.0059    -1.32
retention_7   -0.0082    -4.31


In [3]:
# 1. 가설설정
# 귀무가설 : gate_30과 gate_40의 7일 잔존율은 같다
# 대립가설: 두 버전의 7일 잔존율은 다르다 (양측)

# 2. 실험설계
# 실험 단위: 개인유저를 배정단위로 함.
    # 집단 구성 
    #      ====    대조군 A   |  실험군 B     ====
    # 버전|  gate_30         |   gate_40
    # 처치| 첫 관문 30레벨     |  첫 관문 40 레벨
    # 인원| 44,700           |   45,489
    # 비율| 49.56%           |   50.44%

# 지표 체계 
# 핵심지표 : 7일 잔존율 retention_7
# 보조지표 : 1일 잔존율 retention_1— 즉각적 이탈 반응, 평균 플레이 라운드 avg_rounds  — 참여 깊이, 중앙값 플레이 라운드 median_rounds — 극단값에 강건한 참여 지표

# 가드레일 지표  — 악화 감시용
# 배정 비율(SRM) — 무작위 배정이 제대로 작동했는지, 크래시율 및 로딩 오류율 — 신규 버전의 기술적 결함, 평균 세션 길이 — 과몰입 유발 여부

# 실험기간 : 최소 14일. 7일 잔존율을 관측하려면 마지막으로 유입된 유저도 설치 후 7일이 지나야 하므로 유입 기간 7일에 관측 기간 7일을 더한다.
# 필요 표본 크기 : 현재 7일 잔존율이 약 19%인 상황에서 1.0%p 차이를 유의수준 0.05, 검정력 0.80으로 탐지하려면 집단당 약 2만 5천 명이 필요함.

# 실험 실행 
# 실행 전 중복 배정 점검, 결측치 점검, 배정 무결성(SRM) 점검, 이상치 점검, 지표 정의 정합성 점검

# 결과 분석 계획
# 7일 잔존율 - 2표본 비율검정 또는 카이제곱 독립성 검정
# 1일 잔존율 - 위와 동일
# 평균 플레이 라운드 - 로그 변환 후 t검정 (평균 52.46 대 중앙값 17 격차 때문에 평균 비교보다 중앙값 비교가 적절)

# SUTVA 위반 및 실험 간 간섭 가능성 검토
# 1. 소셜 기능을 통한 간섭 - Cookie Cats에는 친구 초대, 하트 선물, 친구 랭킹 기능이 존재한다. 친구의 진행 상황이 내 플레이에 직접 영향을 주는 구조이다.
# 2. 다계정 및 기기 공유 - 한 사람이 여러 계정을 쓰면 양쪽 버전을 모두 경험하게 되어 처치의 단일성이 깨진다.

# 결론(통계적 검정 이전)
# 배정 비율은 gate_30 49.56%, gate_40 50.44%로 계획에 근접했고 중복 ID와 결측치가 모두 0건이어서 데이터 품질은 양호하다.
# 버전별 지표를 재계산한 결과 7일 잔존율은 19.02%에서 18.20%로 0.82%p(상대 4.31%), 1일 잔존율은 44.82%에서 44.23%로 0.59%p(상대 1.32%) 낮아졌다. 
# 평균 플레이 라운드도 52.46회에서 51.30회로 중앙값은 17회에서 16회로 줄어 네 지표 모두 gate_40이 낮은 방향으로 일관되게 나타났다.
# 다만 이는 관측값을 단순 비교한 기술통계이며 통계적 검정을 수행하지 않았으므로 이 차이가 실제 효과인지 우연인지 판단할 수 없다.
# 친구 초대,선물 기능을 통한 간섭 가능성이 아직 검토되지 않아 SUTVA 충족 여부가 불확실하다.
# 따라서 gate_40 전환에 대한 최종 의사결정은 보류하고
# 7일 잔존율 비율검정과 효과 크기·신뢰구간 산출 이상치 민감도 분석을 완료한 뒤 판단한다.

---

## 실습 마무리

- 어떤 문제가 있었는가?
- 어떻게 개선했는가?
- 무엇을 근거로 개선되었다고 판단했는가?

단순한 그룹별 지표 비교에서 놓칠 수 있는 문제와, 실험 단위·사전 지표·배정 비율·간섭 가능성을 명시하면서 설계가 어떻게 개선되었는지 정리하세요.
